# Reto Semana 9: SecureBank Fraud Detection

## Sistema de Detección de Anomalías en Transacciones con NumPy

---

### Contexto del Proyecto

**SecureBank** es uno de los bancos digitales más grandes de México. Has sido contratado como **Analista de Riesgos Junior** para desarrollar un sistema básico de detección de transacciones anómalas que podrían indicar fraude.

```
╔══════════════════════════════════════════════════════════════════╗
║                    SECUREBANK FRAUD DETECTION                    ║
║                Sistema de Detección de Anomalías                 ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║     Transacciones analizadas: 500 por categoría                  ║
║     Categorías: 5 tipos de comercio                              ║
║     Objetivo: Detectar transacciones sospechosas                 ║
║     Métodos: IQR y Z-Score                                       ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
```

###  Objetivos de Aprendizaje

En este reto aplicarás:
- Cálculo de percentiles y cuartiles
- Detección de outliers con método IQR
- Detección de outliers con Z-Score
- Análisis estadístico por categorías
- Generación de reportes de anomalías

---

## Configuración Inicial

Ejecuta esta celda para cargar NumPy y preparar el entorno.

In [ ]:
import numpy as np

# Configuración para reproducibilidad
np.random.seed(2024)

# Configuración de impresión
np.set_printoptions(precision=2, suppress=True)

print("NumPy cargado correctamente")
print(f"   Versión: {np.__version__}")

NumPy cargado correctamente
   Versión: 2.4.4


---

## Datos del Reto

### Estructura de los Datos

```
CATEGORÍAS DE TRANSACCIONES
════════════════════════════════════════════════════════════════

Índice │ Categoría           │ Monto Típico    │ Descripción
───────┼─────────────────────┼─────────────────┼───────────────
   0   │ Supermercados       │ $200 - $2,000   │ Compras diarias
   1   │ Restaurantes        │ $100 - $800     │ Alimentos
   2   │ Gasolineras         │ $300 - $1,500   │ Combustible
   3   │ Tiendas Online      │ $150 - $5,000   │ E-commerce
   4   │ Entretenimiento     │ $50 - $500      │ Cine, streaming

════════════════════════════════════════════════════════════════

TIPOS DE ANOMALÍAS SIMULADAS:
─────────────────────────────────────────────────────────────────
• Montos inusualmente altos (posible fraude)
• Montos inusualmente bajos (posibles pruebas de tarjeta)
• Patrones atípicos por categoría
─────────────────────────────────────────────────────────────────
```

Ejecuta la siguiente celda para generar los datos de transacciones.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                  GENERACIÓN DE DATOS DE TRANSACCIONES
# ═══════════════════════════════════════════════════════════════════

np.random.seed(2024)

# Configuración por categoría
categorias = ['Supermercados', 'Restaurantes', 'Gasolineras', 'Tiendas_Online', 'Entretenimiento']
n_categorias = len(categorias)

# Parámetros de distribución por categoría (media, desv_std)
params_categorias = {
    'Supermercados': (800, 400),
    'Restaurantes': (350, 150),
    'Gasolineras': (700, 250),
    'Tiendas_Online': (1200, 800),
    'Entretenimiento': (200, 100)
}

n_transacciones_por_cat = 500

# Generar transacciones normales por categoría
transacciones = {}
ids_transaccion = {}

for i, cat in enumerate(categorias):
    media, std = params_categorias[cat]
    
    # Generar montos normales
    montos = np.random.normal(media, std, n_transacciones_por_cat)
    montos = np.maximum(montos, 10)  # Mínimo $10
    
    # Inyectar anomalías (aproximadamente 3-5% del total)
    n_anomalias_altas = np.random.randint(8, 15)
    n_anomalias_bajas = np.random.randint(5, 10)
    
    # Anomalías altas (montos sospechosamente grandes)
    indices_altas = np.random.choice(n_transacciones_por_cat, n_anomalias_altas, replace=False)
    montos[indices_altas] = media + np.random.uniform(4, 8, n_anomalias_altas) * std
    
    # Anomalías bajas (posibles pruebas de tarjeta)
    indices_bajas = np.random.choice(
        [i for i in range(n_transacciones_por_cat) if i not in indices_altas],
        n_anomalias_bajas, replace=False
    )
    montos[indices_bajas] = np.random.uniform(1, 15, n_anomalias_bajas)
    
    transacciones[cat] = montos
    ids_transaccion[cat] = np.arange(i * 1000 + 1, i * 1000 + n_transacciones_por_cat + 1)

# Crear arrays consolidados
# Array 2D: (categorías, transacciones)
montos_matriz = np.array([transacciones[cat] for cat in categorias])

# Arrays 1D para análisis global
todos_montos = np.concatenate([transacciones[cat] for cat in categorias])
todas_categorias = np.concatenate([[cat] * n_transacciones_por_cat for cat in categorias])
todos_ids = np.concatenate([ids_transaccion[cat] for cat in categorias])


def caja_top(ancho):
    return "╔" + "═" * ancho + "╗"

def caja_bottom(ancho):
    return "╚" + "═" * ancho + "╝"

def caja_divisor(ancho):
    return "╠" + "═" * ancho + "╣"

def caja_linea(texto, ancho, centrado=False):
    """Construye una línea de caja con padding exacto para mantener el alineado."""
    contenido = texto.center(ancho) if centrado else texto.ljust(ancho)
    return "║" + contenido + "║"


ANCHO_DATOS = 64
print(caja_top(ANCHO_DATOS))
print(caja_linea("DATOS DE TRANSACCIONES GENERADOS", ANCHO_DATOS, centrado=True))
print(caja_divisor(ANCHO_DATOS))
print(caja_linea(f"  montos_matriz    : {montos_matriz.shape}", ANCHO_DATOS))
print(caja_linea("     (filas=categorías, columnas=transacciones)", ANCHO_DATOS))
print(caja_linea("", ANCHO_DATOS))
print(caja_linea(f"  todos_montos     : {len(todos_montos):,} transacciones totales", ANCHO_DATOS))
print(caja_linea(f"  todas_categorias : {len(todas_categorias):,} etiquetas", ANCHO_DATOS))
print(caja_linea(f"  todos_ids        : {len(todos_ids):,} identificadores", ANCHO_DATOS))
print(caja_bottom(ANCHO_DATOS))

print("\nCategorías disponibles:")
for i, cat in enumerate(categorias):
    media, std = params_categorias[cat]
    print(f"   {i}: {cat:20s} (μ=${media:,}, σ=${std})")

╔════════════════════════════════════════════════════════════════╗
║                DATOS DE TRANSACCIONES GENERADOS                ║
╠════════════════════════════════════════════════════════════════╣
║  montos_matriz    : (5, 500)                                   ║
║     (filas=categorías, columnas=transacciones)                 ║
║                                                                ║
║  todos_montos     : 2,500 transacciones totales                ║
║  todas_categorias : 2,500 etiquetas                            ║
║  todos_ids        : 2,500 identificadores                      ║
╚════════════════════════════════════════════════════════════════╝

Categorías disponibles:
   0: Supermercados        (μ=$800, σ=$400)
   1: Restaurantes         (μ=$350, σ=$150)
   2: Gasolineras          (μ=$700, σ=$250)
   3: Tiendas_Online       (μ=$1,200, σ=$800)
   4: Entretenimiento      (μ=$200, σ=$100)


---

## PARTE 1: Análisis Estadístico por Categoría (30 puntos)

### Ejercicio 1.1: Estadísticas Descriptivas (10 puntos)

Calcula las estadísticas básicas para cada categoría de transacción.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#            EJERCICIO 1.1: ESTADÍSTICAS POR CATEGORÍA
# ═══════════════════════════════════════════════════════════════════

print("ESTADÍSTICAS DESCRIPTIVAS POR CATEGORÍA")
print("═" * 80)

# Arrays para almacenar resultados
medias = np.zeros(n_categorias)
medianas = np.zeros(n_categorias)
stds = np.zeros(n_categorias)
minimos = np.zeros(n_categorias)
maximos = np.zeros(n_categorias)

for i, cat in enumerate(categorias):
    datos = montos_matriz[i]  # Transacciones de esta categoría
    
    medias[i] = np.mean(datos)
    medianas[i] = np.median(datos)
    stds[i] = np.std(datos)
    minimos[i] = np.min(datos)
    maximos[i] = np.max(datos)

# Mostrar resultados en tabla
print(f"\n{'Categoría':<20} {'Media':>12} {'Mediana':>12} {'Std':>12} {'Mín':>10} {'Máx':>10}")
print("─" * 80)

for i, cat in enumerate(categorias):
    print(f"{cat:<20} ${medias[i]:>10,.2f} ${medianas[i]:>10,.2f} ${stds[i]:>10,.2f} ${minimos[i]:>8,.2f} ${maximos[i]:>8,.2f}")

ESTADÍSTICAS DESCRIPTIVAS POR CATEGORÍA
════════════════════════════════════════════════════════════════════════════════

Categoría                   Media      Mediana          Std        Mín        Máx
────────────────────────────────────────────────────────────────────────────────
Supermercados        $    853.56 $    797.39 $    576.21 $    1.32 $3,875.95
Restaurantes         $    362.13 $    352.39 $    203.49 $    2.61 $1,543.27
Gasolineras          $    722.50 $    688.70 $    349.05 $    1.13 $2,676.60
Tiendas_Online       $  1,337.98 $  1,158.48 $  1,122.47 $    1.46 $7,131.76
Entretenimiento      $    209.49 $    196.58 $    127.72 $    1.14 $  981.56


### Ejercicio 1.2: Cuartiles e IQR (10 puntos)

Calcula los cuartiles y el rango intercuartílico para cada categoría.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                 EJERCICIO 1.2: CUARTILES E IQR
# ═══════════════════════════════════════════════════════════════════

print("CUARTILES E IQR POR CATEGORÍA")
print("═" * 80)

# Arrays para almacenar resultados
Q1_arr = np.zeros(n_categorias)
Q2_arr = np.zeros(n_categorias)
Q3_arr = np.zeros(n_categorias)
IQR_arr = np.zeros(n_categorias)

for i, cat in enumerate(categorias):
    datos = montos_matriz[i]
    
    Q1_arr[i] = np.percentile(datos, 25)
    Q2_arr[i] = np.percentile(datos, 50)
    Q3_arr[i] = np.percentile(datos, 75)
    IQR_arr[i] = Q3_arr[i] - Q1_arr[i]

# Mostrar resultados
print(f"\n{'Categoría':<20} {'Q1 (25%)':>12} {'Q2 (50%)':>12} {'Q3 (75%)':>12} {'IQR':>12}")
print("─" * 72)

for i, cat in enumerate(categorias):
    print(f"{cat:<20} ${Q1_arr[i]:>10,.2f} ${Q2_arr[i]:>10,.2f} ${Q3_arr[i]:>10,.2f} ${IQR_arr[i]:>10,.2f}")

CUARTILES E IQR POR CATEGORÍA
════════════════════════════════════════════════════════════════════════════════

Categoría                Q1 (25%)     Q2 (50%)     Q3 (75%)          IQR
────────────────────────────────────────────────────────────────────────
Supermercados        $    510.27 $    797.39 $  1,103.41 $    593.14
Restaurantes         $    242.13 $    352.39 $    449.41 $    207.29
Gasolineras          $    511.29 $    688.70 $    881.19 $    369.90
Tiendas_Online       $    663.81 $  1,158.48 $  1,809.95 $  1,146.14
Entretenimiento      $    133.53 $    196.58 $    268.68 $    135.15


### Ejercicio 1.3: Límites para Outliers (10 puntos)

Calcula los límites inferior y superior para detectar outliers usando el método IQR.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#               EJERCICIO 1.3: LÍMITES PARA OUTLIERS
# ═══════════════════════════════════════════════════════════════════

print("LÍMITES PARA DETECCIÓN DE OUTLIERS (Método IQR)")
print("═" * 80)

# Factor IQR estándar
FACTOR_IQR = 1.5

# Arrays para almacenar límites
limites_inf = np.zeros(n_categorias)
limites_sup = np.zeros(n_categorias)

for i, cat in enumerate(categorias):
    # Límite inferior = Q1 - 1.5 * IQR
    # Límite superior = Q3 + 1.5 * IQR
    limites_inf[i] = Q1_arr[i] - FACTOR_IQR * IQR_arr[i]
    limites_sup[i] = Q3_arr[i] + FACTOR_IQR * IQR_arr[i]

# Mostrar resultados
print(f"\n{'Categoría':<20} {'Límite Inf':>15} {'Límite Sup':>15} {'Rango Válido':>20}")
print("─" * 75)

for i, cat in enumerate(categorias):
    # El límite inferior no puede ser negativo para montos
    lim_inf_real = max(0, limites_inf[i])
    rango = f"${lim_inf_real:,.0f} - ${limites_sup[i]:,.0f}"
    print(f"{cat:<20} ${limites_inf[i]:>13,.2f} ${limites_sup[i]:>13,.2f} {rango:>20}")

LÍMITES PARA DETECCIÓN DE OUTLIERS (Método IQR)
════════════════════════════════════════════════════════════════════════════════

Categoría                 Límite Inf      Límite Sup         Rango Válido
───────────────────────────────────────────────────────────────────────────
Supermercados        $      -379.45 $     1,993.12          $0 - $1,993
Restaurantes         $       -68.81 $       760.35            $0 - $760
Gasolineras          $       -43.56 $     1,436.03          $0 - $1,436
Tiendas_Online       $    -1,055.40 $     3,529.16          $0 - $3,529
Entretenimiento      $       -69.19 $       471.40            $0 - $471


---

## PARTE 2: Detección de Outliers con IQR (25 puntos)

### Ejercicio 2.1: Identificar Outliers por Categoría (15 puntos)

Detecta las transacciones anómalas en cada categoría usando el método IQR.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#            EJERCICIO 2.1: DETECCIÓN DE OUTLIERS CON IQR
# ═══════════════════════════════════════════════════════════════════

print("DETECCIÓN DE TRANSACCIONES ANÓMALAS (Método IQR)")
print("═" * 80)

# Diccionario para almacenar resultados
outliers_iqr = {}
n_outliers_iqr = np.zeros(n_categorias, dtype=int)

for i, cat in enumerate(categorias):
    datos = montos_matriz[i]
    ids = ids_transaccion[cat]
    
    # Un valor es outlier si: valor < limite_inferior OR valor > limite_superior
    mascara_outliers = (datos < limites_inf[i]) | (datos > limites_sup[i])
    
    # Separar outliers inferiores y superiores
    mascara_inf = datos < limites_inf[i]
    mascara_sup = datos > limites_sup[i]
    
    # Almacenar resultados
    outliers_iqr[cat] = {
        'ids': ids[mascara_outliers],
        'montos': datos[mascara_outliers],
        'n_total': np.sum(mascara_outliers),
        'n_inferiores': np.sum(mascara_inf),
        'n_superiores': np.sum(mascara_sup)
    }
    n_outliers_iqr[i] = np.sum(mascara_outliers)

# Mostrar resumen
print(f"\n{'Categoría':<20} {'Total Trans.':>12} {'Outliers':>10} {'% Anomalías':>12} {'Inf.':>8} {'Sup.':>8}")
print("─" * 75)

for i, cat in enumerate(categorias):
    pct = (n_outliers_iqr[i] / n_transacciones_por_cat) * 100
    info = outliers_iqr[cat]
    print(f"{cat:<20} {n_transacciones_por_cat:>12,} {info['n_total']:>10} {pct:>11.1f}% {info['n_inferiores']:>8} {info['n_superiores']:>8}")

print(f"\nTotal de outliers detectados: {np.sum(n_outliers_iqr)}")

DETECCIÓN DE TRANSACCIONES ANÓMALAS (Método IQR)
════════════════════════════════════════════════════════════════════════════════

Categoría            Total Trans.   Outliers  % Anomalías     Inf.     Sup.
───────────────────────────────────────────────────────────────────────────
Supermercados                 500         15         3.0%        0       15
Restaurantes                  500         16         3.2%        0       16
Gasolineras                   500         13         2.6%        0       13
Tiendas_Online                500         14         2.8%        0       14
Entretenimiento               500         13         2.6%        0       13

Total de outliers detectados: 71


### Ejercicio 2.2: Análisis de Outliers Detectados (10 puntos)

Analiza las características de los outliers detectados.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#              EJERCICIO 2.2: ANÁLISIS DE OUTLIERS IQR
# ═══════════════════════════════════════════════════════════════════

print("ANÁLISIS DETALLADO DE OUTLIERS (Método IQR)")
print("═" * 80)

for cat in categorias:
    info = outliers_iqr[cat]
    
    if info['n_total'] > 0:
        montos_out = info['montos']
        
        monto_min_outlier = np.min(montos_out)
        monto_max_outlier = np.max(montos_out)
        monto_promedio_outlier = np.mean(montos_out)
        
        print(f"\n{cat}")
        print(f"   Outliers detectados: {info['n_total']}")
        print(f"   Monto mínimo outlier: ${monto_min_outlier:,.2f}")
        print(f"   Monto máximo outlier: ${monto_max_outlier:,.2f}")
        print(f"   Monto promedio outlier: ${monto_promedio_outlier:,.2f}")
        
        # Mostrar los 3 outliers más extremos (más altos)
        if info['n_superiores'] > 0:
            idx_ordenados = np.argsort(montos_out)[::-1]  # Descendente
            print(f"   Top 3 montos más altos:")
            for j in range(min(3, len(idx_ordenados))):
                idx = idx_ordenados[j]
                print(f"      - ID {info['ids'][idx]}: ${montos_out[idx]:,.2f}")

ANÁLISIS DETALLADO DE OUTLIERS (Método IQR)
════════════════════════════════════════════════════════════════════════════════

Supermercados
   Outliers detectados: 15
   Monto mínimo outlier: $2,027.97
   Monto máximo outlier: $3,875.95
   Monto promedio outlier: $3,128.03
   Top 3 montos más altos:
      - ID 29: $3,875.95
      - ID 117: $3,796.12
      - ID 473: $3,794.56

Restaurantes
   Outliers detectados: 16
   Monto mínimo outlier: $771.49
   Monto máximo outlier: $1,543.27
   Monto promedio outlier: $1,114.75
   Top 3 montos más altos:
      - ID 1079: $1,543.27
      - ID 1057: $1,396.68
      - ID 1304: $1,345.08

Gasolineras
   Outliers detectados: 13
   Monto mínimo outlier: $1,445.44
   Monto máximo outlier: $2,676.60
   Monto promedio outlier: $2,080.94
   Top 3 montos más altos:
      - ID 2066: $2,676.60
      - ID 2367: $2,585.56
      - ID 2038: $2,514.10

Tiendas_Online
   Outliers detectados: 14
   Monto mínimo outlier: $3,532.38
   Monto máximo outlier: $7,131.76


---

## PARTE 3: Detección de Outliers con Z-Score (25 puntos)

### Ejercicio 3.1: Calcular Z-Scores (10 puntos)

Calcula el Z-Score para cada transacción.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#               EJERCICIO 3.1: CÁLCULO DE Z-SCORES
# ═══════════════════════════════════════════════════════════════════

print("CÁLCULO DE Z-SCORES POR CATEGORÍA")
print("═" * 80)

# Umbral para considerar outlier
UMBRAL_ZSCORE = 3

# Matriz para almacenar z-scores
zscores_matriz = np.zeros_like(montos_matriz)

for i, cat in enumerate(categorias):
    datos = montos_matriz[i]
    
    # Fórmula: z = (x - media) / std
    media_cat = np.mean(datos)
    std_cat = np.std(datos)
    zscores_matriz[i] = (datos - media_cat) / std_cat

# Verificar cálculo mostrando estadísticas de z-scores
print(f"\n{'Categoría':<20} {'Media Z':>10} {'Std Z':>10} {'Min Z':>10} {'Max Z':>10}")
print("─" * 65)

for i, cat in enumerate(categorias):
    zs = zscores_matriz[i]
    print(f"{cat:<20} {np.mean(zs):>10.4f} {np.std(zs):>10.4f} {np.min(zs):>10.2f} {np.max(zs):>10.2f}")

print(f"\nNota: La media de Z-scores debe ser ~0 y la std ~1")

CÁLCULO DE Z-SCORES POR CATEGORÍA
════════════════════════════════════════════════════════════════════════════════

Categoría               Media Z      Std Z      Min Z      Max Z
─────────────────────────────────────────────────────────────────
Supermercados           -0.0000     1.0000      -1.48       5.25
Restaurantes             0.0000     1.0000      -1.77       5.80
Gasolineras              0.0000     1.0000      -2.07       5.60
Tiendas_Online          -0.0000     1.0000      -1.19       5.16
Entretenimiento         -0.0000     1.0000      -1.63       6.05

Nota: La media de Z-scores debe ser ~0 y la std ~1


### Ejercicio 3.2: Detectar Outliers con Z-Score (15 puntos)

Identifica outliers donde |Z-Score| > 3.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#          EJERCICIO 3.2: DETECCIÓN DE OUTLIERS CON Z-SCORE
# ═══════════════════════════════════════════════════════════════════

print(f"DETECCIÓN DE OUTLIERS CON Z-SCORE (umbral = {UMBRAL_ZSCORE})")
print("═" * 80)

# Diccionario para almacenar resultados
outliers_zscore = {}
n_outliers_zscore = np.zeros(n_categorias, dtype=int)

for i, cat in enumerate(categorias):
    datos = montos_matriz[i]
    zscores = zscores_matriz[i]
    ids = ids_transaccion[cat]
    
    # Outliers donde |z-score| > umbral
    mascara_outliers_z = np.abs(zscores) > UMBRAL_ZSCORE
    
    # Clasificar por tipo
    mascara_z_neg = zscores < -UMBRAL_ZSCORE  # Muy bajos
    mascara_z_pos = zscores > UMBRAL_ZSCORE   # Muy altos
    
    outliers_zscore[cat] = {
        'ids': ids[mascara_outliers_z],
        'montos': datos[mascara_outliers_z],
        'zscores': zscores[mascara_outliers_z],
        'n_total': np.sum(mascara_outliers_z),
        'n_bajos': np.sum(mascara_z_neg),
        'n_altos': np.sum(mascara_z_pos)
    }
    n_outliers_zscore[i] = np.sum(mascara_outliers_z)

# Mostrar resumen
print(f"\n{'Categoría':<20} {'Total Trans.':>12} {'Outliers':>10} {'% Anomalías':>12} {'Z<-3':>8} {'Z>3':>8}")
print("─" * 75)

for i, cat in enumerate(categorias):
    pct = (n_outliers_zscore[i] / n_transacciones_por_cat) * 100
    info = outliers_zscore[cat]
    print(f"{cat:<20} {n_transacciones_por_cat:>12,} {info['n_total']:>10} {pct:>11.1f}% {info['n_bajos']:>8} {info['n_altos']:>8}")

print(f"\nTotal de outliers detectados (Z-Score): {np.sum(n_outliers_zscore)}")

DETECCIÓN DE OUTLIERS CON Z-SCORE (umbral = 3)
════════════════════════════════════════════════════════════════════════════════

Categoría            Total Trans.   Outliers  % Anomalías     Z<-3      Z>3
───────────────────────────────────────────────────────────────────────────
Supermercados                 500         12         2.4%        0       12
Restaurantes                  500         11         2.2%        0       11
Gasolineras                   500         10         2.0%        0       10
Tiendas_Online                500         13         2.6%        0       13
Entretenimiento               500          9         1.8%        0        9

Total de outliers detectados (Z-Score): 55


---

## PARTE 4: Comparación y Reporte Final (20 puntos)

### Ejercicio 4.1: Comparar Métodos (10 puntos)

Compara los resultados de ambos métodos de detección.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#               EJERCICIO 4.1: COMPARACIÓN DE MÉTODOS
# ═══════════════════════════════════════════════════════════════════

print("COMPARACIÓN DE MÉTODOS DE DETECCIÓN")
print("═" * 80)

# Total de outliers por método
total_iqr = np.sum(n_outliers_iqr)
total_zscore = np.sum(n_outliers_zscore)

print(f"\nRESUMEN GLOBAL:")
print(f"   Método IQR:     {total_iqr} outliers detectados")
print(f"   Método Z-Score: {total_zscore} outliers detectados")

# Comparación por categoría
print(f"\n{'Categoría':<20} {'IQR':>10} {'Z-Score':>10} {'Diferencia':>12} {'Coincidencia':>15}")
print("─" * 72)

for i, cat in enumerate(categorias):
    n_iqr = n_outliers_iqr[i]
    n_zs = n_outliers_zscore[i]
    diff = n_iqr - n_zs
    
    # Cuántos outliers coinciden en ambos métodos (están en ambas listas)
    ids_iqr = set(outliers_iqr[cat]['ids'])
    ids_zscore = set(outliers_zscore[cat]['ids'])
    coincidentes = len(ids_iqr & ids_zscore)  # Intersección
    
    print(f"{cat:<20} {n_iqr:>10} {n_zs:>10} {diff:>+12} {coincidentes:>15}")

COMPARACIÓN DE MÉTODOS DE DETECCIÓN
════════════════════════════════════════════════════════════════════════════════

RESUMEN GLOBAL:
   Método IQR:     71 outliers detectados
   Método Z-Score: 55 outliers detectados

Categoría                   IQR    Z-Score   Diferencia    Coincidencia
────────────────────────────────────────────────────────────────────────
Supermercados                15         12           +3              12
Restaurantes                 16         11           +5              11
Gasolineras                  13         10           +3              10
Tiendas_Online               14         13           +1              13
Entretenimiento              13          9           +4               9


### Ejercicio 4.2: Reporte de Transacciones Sospechosas (10 puntos)

Genera un reporte final de las transacciones más sospechosas.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#           EJERCICIO 4.2: REPORTE DE TRANSACCIONES SOSPECHOSAS
# ═══════════════════════════════════════════════════════════════════

ANCHO_REPORTE = 74
print(caja_top(ANCHO_REPORTE))
print(caja_linea("", ANCHO_REPORTE))
print(caja_linea("SECUREBANK - REPORTE DE TRANSACCIONES SOSPECHOSAS", ANCHO_REPORTE, centrado=True))
print(caja_linea("", ANCHO_REPORTE))
print(caja_divisor(ANCHO_REPORTE))
print(caja_linea("", ANCHO_REPORTE))
print(caja_linea("  ALTA PRIORIDAD (detectadas por ambos métodos)", ANCHO_REPORTE))
print(caja_linea("  " + "-" * (ANCHO_REPORTE - 4), ANCHO_REPORTE))

total_alta_prioridad = 0

for cat in categorias:
    ids_iqr = set(outliers_iqr[cat]['ids'])
    ids_zscore = set(outliers_zscore[cat]['ids'])
    
    # Transacciones que están en AMBOS conjuntos
    ids_ambos = ids_iqr & ids_zscore
    
    if len(ids_ambos) > 0:
        total_alta_prioridad += len(ids_ambos)
        
        # Obtener montos de estas transacciones
        for trans_id in list(ids_ambos)[:3]:  # Mostrar máximo 3
            idx = np.where(ids_transaccion[cat] == trans_id)[0][0]
            monto = montos_matriz[categorias.index(cat), idx]
            zscore = zscores_matriz[categorias.index(cat), idx]
            linea = f"    ID {trans_id}: {cat:15s} ${monto:>10,.2f} (Z={zscore:+.2f})"
            print(caja_linea(linea, ANCHO_REPORTE))

print(caja_linea("", ANCHO_REPORTE))
print(caja_linea("  RESUMEN EJECUTIVO", ANCHO_REPORTE))
print(caja_linea("  " + "-" * (ANCHO_REPORTE - 4), ANCHO_REPORTE))

# Calcula estadísticas finales
total_transacciones = len(todos_montos)
ids_outliers_iqr_todos = set(np.concatenate([outliers_iqr[cat]['ids'] for cat in categorias]))
ids_outliers_zscore_todos = set(np.concatenate([outliers_zscore[cat]['ids'] for cat in categorias]))
total_outliers_unicos = len(ids_outliers_iqr_todos | ids_outliers_zscore_todos)  # Unión de ambos métodos
pct_anomalias = (total_outliers_unicos / total_transacciones) * 100

print(caja_linea(f"    Total transacciones analizadas:    {total_transacciones:>6,}", ANCHO_REPORTE))
print(caja_linea(f"    Transacciones sospechosas:         {total_outliers_unicos:>6,} ({pct_anomalias:.1f}%)", ANCHO_REPORTE))
print(caja_linea(f"    Alta prioridad (ambos métodos):    {total_alta_prioridad:>6,}", ANCHO_REPORTE))
print(caja_linea("", ANCHO_REPORTE))
print(caja_bottom(ANCHO_REPORTE))

╔══════════════════════════════════════════════════════════════════════════╗
║                                                                          ║
║            SECUREBANK - REPORTE DE TRANSACCIONES SOSPECHOSAS             ║
║                                                                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  ALTA PRIORIDAD (detectadas por ambos métodos)                           ║
║  ----------------------------------------------------------------------  ║
║    ID 356: Supermercados   $  2,743.01 (Z=+3.28)                         ║
║    ID 249: Supermercados   $  3,719.03 (Z=+4.97)                         ║
║    ID 390: Supermercados   $  2,676.41 (Z=+3.16)                         ║
║    ID 1089: Restaurantes    $  1,133.24 (Z=+3.79)                        ║
║    ID 1057: Restaurantes    $  1,396.68 (Z=+5.08)                        ║

---

## 🎯 EJERCICIO BONUS: Análisis de Correlación (10 puntos)

Analiza si existe correlación entre categorías en los patrones de gasto.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#                  EJERCICIO BONUS: CORRELACIÓN
# ═══════════════════════════════════════════════════════════════════

print("ANÁLISIS DE CORRELACIÓN ENTRE CATEGORÍAS")
print("═" * 70)

# Matriz de correlación entre categorías (cada fila de montos_matriz es una categoría)
matriz_correlacion = np.corrcoef(montos_matriz)

# Mostrar matriz de correlación
print(f"\n{'':>18}", end='')
for cat in categorias:
    print(f"{cat[:8]:>10}", end='')
print()
print("─" * 70)

for i, cat in enumerate(categorias):
    print(f"{cat:<18}", end='')
    for j in range(n_categorias):
        valor = matriz_correlacion[i, j]
        if i == j:
            print(f"{'1.00':>10}", end='')
        else:
            print(f"{valor:>10.3f}", end='')
    print()

# Encontrar las correlaciones más fuertes (excluyendo diagonal)
print(f"\nCORRELACIONES MÁS FUERTES:")
print("─" * 50)

# Identifica el par de categorías con mayor correlación (excluyendo la diagonal)
max_corr = 0
par_max = ('', '')

for i in range(n_categorias):
    for j in range(i+1, n_categorias):
        if abs(matriz_correlacion[i, j]) > abs(max_corr):
            max_corr = matriz_correlacion[i, j]
            par_max = (categorias[i], categorias[j])

print(f"   Mayor correlación: {par_max[0]} <-> {par_max[1]}")
print(f"   Valor: {max_corr:.4f}")

ANÁLISIS DE CORRELACIÓN ENTRE CATEGORÍAS
══════════════════════════════════════════════════════════════════════

                    Supermer  Restaura  Gasoline  Tiendas_  Entreten
──────────────────────────────────────────────────────────────────────
Supermercados           1.00    -0.040     0.033    -0.029    -0.024
Restaurantes          -0.040      1.00    -0.104    -0.060     0.035
Gasolineras            0.033    -0.104      1.00    -0.042     0.029
Tiendas_Online        -0.029    -0.060    -0.042      1.00    -0.077
Entretenimiento       -0.024     0.035     0.029    -0.077      1.00

CORRELACIONES MÁS FUERTES:
──────────────────────────────────────────────────
   Mayor correlación: Restaurantes <-> Gasolineras
   Valor: -0.1038


---

## ✅ Checklist de Entrega

Antes de entregar, verifica que:

| # | Criterio | Puntos |
|---|----------|--------|
| 1 | Parte 1: Análisis Estadístico por Categoría | 30 |
| 2 | Parte 2: Detección de Outliers con IQR | 25 |
| 3 | Parte 3: Detección de Outliers con Z-Score | 25 |
| 4 | Parte 4: Comparación y Reporte Final | 20 |
| 5 | BONUS: Análisis de Correlación | +10 |
| | **Total posible** | **110** |

### 📝 Notas Importantes

- Usa operaciones vectorizadas de NumPy (evita loops donde sea posible)
- Verifica que tus Z-Scores tengan media ~0 y std ~1
- El código debe ejecutarse sin errores de principio a fin
- Los outliers detectados deben ser razonables (3-5% típicamente)

---

### 🏅 Criterios de Evaluación

```
RÚBRICA DE CALIFICACIÓN
═══════════════════════════════════════════════════════

✓ Cálculos estadísticos correctos         → 40%
✓ Detección de outliers precisa           → 30%
✓ Comparación de métodos coherente        → 15%
✓ Código limpio y vectorizado             → 10%
✓ Interpretaciones razonables             → 5%

═══════════════════════════════════════════════════════
```

---

*SecureBank Fraud Detection - Reto desarrollado para Programación para Ciencia de Datos, IPN 2026*